In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('dirty_cafe_sales.csv')
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


### discovering the data

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [4]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [5]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [6]:
df['Item'].value_counts()

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
ERROR        292
Name: count, dtype: int64

In [7]:
df['Payment Method'].value_counts()

Payment Method
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

In [8]:
df['Location'].value_counts()

Location
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

### string colunms cleaning
- we can't delete error or unknown values because they are much and deleting them makes our data smaller and less accurate
- we can't impute them with mean because it will create a fake winner
- so we will replace "null","error" and "unkown" values with one value called missing


In [9]:
cols = ['Item', 'Payment Method', 'Location']

df[cols] = df[cols].replace(['UNKNOWN', 'ERROR'], 'Missing')

In [10]:
df['Item'].value_counts()

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
Missing      636
Name: count, dtype: int64

In [11]:
df['Item'].isna().sum()

np.int64(333)

In [12]:
df[cols].isna().sum()

Item               333
Payment Method    2579
Location          3265
dtype: int64

In [13]:
df[cols] = df[cols].fillna('Missing')

In [14]:
df[cols].isna().sum()

Item              0
Payment Method    0
Location          0
dtype: int64

In [15]:
df.isna().sum()

Transaction ID        0
Item                  0
Quantity            138
Price Per Unit      179
Total Spent         173
Payment Method        0
Location              0
Transaction Date    159
dtype: int64

### neumric columns cleaning
- first we will change data type to neumric then we will perform a mathmatical opreatoin to handle missing values : 
total spent = price per unit * quantity


In [16]:
cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])
df['Quantity'] = df['Quantity'].fillna(df['Total Spent'] / df['Price Per Unit'])
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])

In [17]:
df.isna().sum()

Transaction ID        0
Item                  0
Quantity             38
Price Per Unit       38
Total Spent          40
Payment Method        0
Location              0
Transaction Date    159
dtype: int64

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    10000 non-null  object 
 1   Item              10000 non-null  object 
 2   Quantity          9962 non-null   float64
 3   Price Per Unit    9962 non-null   float64
 4   Total Spent       9960 non-null   float64
 5   Payment Method    10000 non-null  object 
 6   Location          10000 non-null  object 
 7   Transaction Date  9841 non-null   object 
dtypes: float64(3), object(5)
memory usage: 625.1+ KB


In [ ]:
cols = ['Quantity', 'Price Per Unit', 'Total Spent']
for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[cols] = df[cols].fillna(df[cols].mean())

In [39]:
df['Total Spent'] = df['Quantity'] * df['Price Per Unit']

In [28]:
df.isna().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    159
dtype: int64

### cleaning date colunms
- first we will change data type to date
- We will apply the forward fill (ffill) technique to handle missing dates, assuming that missing entries belong to the same period as the preceding record.

In [29]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              10000 non-null  object        
 2   Quantity          10000 non-null  float64       
 3   Price Per Unit    10000 non-null  float64       
 4   Total Spent       10000 non-null  float64       
 5   Payment Method    10000 non-null  object        
 6   Location          10000 non-null  object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


In [31]:
df['Transaction Date'].value_counts()

Transaction Date
2023-06-16    40
2023-02-06    40
2023-09-21    39
2023-03-13    39
2023-07-24    39
              ..
2023-11-24    15
2023-04-27    15
2023-07-22    14
2023-03-11    14
2023-02-17    14
Name: count, Length: 365, dtype: int64

In [33]:
df['Transaction Date'] = df['Transaction Date'].ffill()

In [34]:
df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

add a season column

In [35]:
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

In [37]:
df['Season'] = df['Transaction Date'].apply(get_season)

In [41]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Season
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,Autumn
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,Spring
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,Summer
3,TXN_7034554,Salad,2.0,5.0,10.0,Missing,Missing,2023-04-27,Spring
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,Summer


save  the cleaned data to csv file

In [42]:
df.to_csv('cleaned_cafe_sales.csv')